In [1]:
# @title Kaggle
from google.colab import userdata
from pathlib import Path

def create_key(key):
  key_path = Path("/root/.config/kaggle/kaggle.json")
  key_path.parent.mkdir(parents=True, exist_ok=True)

  with open(key_path, 'w') as file:
    file.write(key)

  key_path.chmod(600)

try:
  key = userdata.get('kaggle-api')
  create_key(key)
except userdata.SecretNotFoundError:
  print('Get API token from kaggle > setting > api > create new token and store the file contect in the google colab secret named "kaggle-api"')

In [ ]:
# @title Download Dataset
import kaggle
from pathlib import Path

dataset_loc = Path("movie-dataset")

kaggle.api.dataset_download_files('rounakbanik/the-movies-dataset', path=dataset_loc, unzip=True)

In [ ]:
# @title Load Data
import pandas as pd
metadata_file = dataset_loc / 'movies_metadata.csv'
metadata = pd.read_csv(metadata_file)

In [8]:
# @title Process
import json
from pprint import pprint

processed = {
    'movies': [],
    'genres': {}
}

processed_json = 'movie_metadata.json'
raw_data_file = 'movie_metadata.txt'

for i, row in metadata.iterrows():
  genres = eval(row['genres'])
  for g in genres:
    processed['genres'][g['id']] = g['name']
  movie = {
      'genres': [g['id'] for g in genres],
      'id': row['id'],
      'imdb_id': row['imdb_id'],
      'overview': row['overview'],
      'title': row['title'],
      'runtime': row['runtime'],
      'vote_average': row['vote_average'],
      'vote_count': row['vote_count'],
      'release': row['release_date'],
  }
  processed['movies'].append(movie)

with open(processed_json, 'w') as file:
  json.dump(processed, file)

with open(raw_data_file, 'w', encoding="utf-8") as file:
  file.write(f"genres start\n")
  for key, val in processed['genres'].items():
    file.write(f"{key}\n{val}\n")
  file.write(f"genres end\n")
  movie_keys = list(processed['movies'][0].keys())
  file.write(f"movies start\n")
  for movie in processed['movies']:
    for key in movie_keys:
      val = movie[key]
      if isinstance(val, list):
        val = " ".join(map(str, val))
      file.write(f"{val}\n")
  file.write(f"movies end\n")



In [9]:
print(list(processed['movies'][0].keys()))

['genres', 'id', 'imdb_id', 'overview', 'title', 'runtime', 'vote_average', 'vote_count', 'release']
